# *Cut tiles*

In [26]:
import cv2
import os

def crop_image_and_save(image_path, output_folder, tile_size=640):
    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Read the original image
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Cannot read image at {image_path}")
    
    height, width = image.shape[:2]
    tile_count = 0

    # Split the image into tiles of size tile_size x tile_size
    for y in range(0, height, tile_size):
        for x in range(0, width, tile_size):
            # Crop a section of the image
            tile = image[y:y+tile_size, x:x+tile_size]
            
            # Save the cropped tile to the output folder
            tile_filename = f"tile_{tile_count:03d}.tif"
            tile_path = os.path.join(output_folder, tile_filename)
            cv2.imwrite(tile_path, tile)
            tile_count += 1

    print(f"Saved {tile_count} tiles in folder {output_folder}")

# Specify the path to the original image and the output folder
image_path = "D:/SIG/Pantanaw/Test_image/pantanaw_03.tif"
output_folder = "D:/SIG/Pantanaw/Test_image/pantanaw_03_604"
# Crop the image and save the tiles
crop_image_and_save(image_path, output_folder, tile_size=640)


Saved 12 tiles in folder D:/SIG/Pantanaw/Test_image/pantanaw_03_604


# *Cut tils and Yolo*

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# ฟังก์ชั่นตัดภาพเป็น tiles
def crop_image(image, tile_size=640):
    tiles = []
    height, width = image.shape[:2]
    
    # ตัดภาพเป็น tiles ขนาด 640x640
    for y in range(0, height, tile_size):
        for x in range(0, width, tile_size):
            # ตัดส่วนภาพให้มีขนาดเท่ากับ tile_size
            tile = image[y:y+tile_size, x:x+tile_size]
            tiles.append((tile, x, y))  # เก็บทั้ง tile, x_offset, y_offset
    return tiles

# ฟังก์ชั่นรวมผลลัพธ์การทำนายจากแต่ละ tile
def combine_predicted_tiles(tiles, tile_size, image_shape):
    combined_image = np.zeros((image_shape[0], image_shape[1], 3), dtype=np.uint8)  # สร้างภาพเปล่าขนาดใหญ่เป็นภาพสี
    
    for tile, x_offset, y_offset in tiles:
        tile_height, tile_width = tile.shape[:2]
        # วาง tile ที่ทำนายแล้วกลับที่เดิมในภาพใหญ่
        combined_image[y_offset:y_offset+tile_height, x_offset:x_offset+tile_width] = tile
    
    return combined_image

# ฟังก์ชั่นใช้ YOLO ทำนาย bounding boxes
def process_yolo_results(image, tiles, tile_size=640):
    yolo_model = YOLO("D:/SIG/Yolo/training_results/field_detection_exp1/weights/best.pt")  # ใส่ที่อยู่ของไฟล์โมเดล YOLO
    image_height, image_width = image.shape[:2]
    predicted_tiles = []
    
    for tile, x_offset, y_offset in tiles:
        # ทำนายด้วย YOLO สำหรับแต่ละ tile
        yolo_results = yolo_model.predict(source=tile, save=False)  # ไม่ต้องบันทึกเป็นไฟล์
        
        # ดึงภาพที่ทำนายแล้วจากผลลัพธ์ของ YOLO
        predicted_tile = yolo_results[0].plot()  # ใช้ plot() เพื่อได้ภาพที่มี bounding boxes
        
        # แปลง predicted_tile ให้เป็น numpy array ก่อนนำมารวม
        predicted_tile = np.array(predicted_tile) 
        
        # แปลงให้เป็นภาพสี (ถ้าต้องการเป็นสี)
        if len(predicted_tile.shape) == 2:  # ถ้าเป็นภาพขาวดำ
            predicted_tile = cv2.cvtColor(predicted_tile, cv2.COLOR_GRAY2BGR)
        
        predicted_tiles.append((predicted_tile, x_offset, y_offset))
    
    # รวมภาพที่ทำนายแล้วทั้งหมด
    combined_image = combine_predicted_tiles(predicted_tiles, tile_size, (image_height, image_width))
    
    return combined_image

# อ่านภาพต้นฉบับ
image_path = "D:/SIG/Pantanaw/Test_image/pantanaw_01.tif"
image = cv2.imread(image_path)

# ตัดภาพเป็น tiles ขนาด 640x640
tiles = crop_image(image, tile_size=640)

# ประมวลผลผลลัพธ์ YOLO
combined_image = process_yolo_results(image, tiles, tile_size=640)

# แสดงภาพผลลัพธ์
cv2.imshow("Detected Image", combined_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# หรือบันทึกภาพผลลัพธ์
cv2.imwrite("D:/SIG/Pantanaw/Test_image/pantanaw_640/pantanaw_01_02_with_bboxes.tif", combined_image)
cv2.imwrite("D:/SIG/Pantanaw/Test_image/pantanaw_640/pantanaw_01_02_with_bboxes.jpg", combined_image)



0: 640x640 5 fields, 116.3ms
Speed: 0.0ms preprocess, 116.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 11 fields, 99.2ms
Speed: 17.6ms preprocess, 99.2ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 field, 99.7ms
Speed: 17.7ms preprocess, 99.7ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 4 fields, 84.9ms
Speed: 14.3ms preprocess, 84.9ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 7 fields, 94.8ms
Speed: 4.0ms preprocess, 94.8ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x320 4 fields, 89.2ms
Speed: 15.0ms preprocess, 89.2ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 320)

0: 640x640 3 fields, 110.4ms
Speed: 0.0ms preprocess, 110.4ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 fields, 106.4ms
Speed: 0.0ms preprocess, 106.4ms inference, 0.0ms postprocess per image at shape

True